# Enable MCP Flag on HTTP Connection

Use this notebook only if the **Is MCP connection** toggle was missing from the UI when you created the connection. Complete Steps 1 and 2 of `MCP-MANUAL-SETUP.md` first; the HTTP connection must already exist.

The SDK `update` call replaces the full options map. Because the API never returns `client_secret` on a GET, every option including the secret must be re-provided here.

After running all three cells, proceed to `mcp-validate.ipynb` to confirm the connection works.

In [ ]:
# Connection name chosen in Step 1 of MCP-MANUAL-SETUP.md
CONNECTION_NAME = "neo4j_aircraft_mcp_server"

# Values from neo4j-agentcore-mcp-server/.mcp-credentials.json
# Split gateway_url into HOST (scheme + domain) and BASE_PATH (path only).
HOST           = "https://<gateway-host>.gateway.bedrock-agentcore.<region>.amazonaws.com"
PORT           = "443"
BASE_PATH      = "/mcp"
CLIENT_ID      = "<client_id>"
CLIENT_SECRET  = "<client_secret>"
OAUTH_SCOPE    = "<scope>"
TOKEN_ENDPOINT = "https://<cognito-domain>/oauth2/token"

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

w.connections.update(
    name=CONNECTION_NAME,
    options={
        "host":              HOST,
        "port":              PORT,
        "base_path":         BASE_PATH,
        "client_id":         CLIENT_ID,
        "client_secret":     CLIENT_SECRET,
        "oauth_scope":       OAUTH_SCOPE,
        "token_endpoint":    TOKEN_ENDPOINT,
        "is_mcp_connection": "true",
    },
)

print(f"Updated '{CONNECTION_NAME}'.")

In [ ]:
conn   = w.connections.get(CONNECTION_NAME)
opts   = conn.options or {}
is_mcp = opts.get("is_mcp_connection", "false")
url    = conn.url or ""

print(f"is_mcp_connection : {is_mcp}")
print(f"url               : {url}")

if is_mcp == "true" and url.endswith("/mcp"):
    print("\nReady. Proceed to mcp-validate.ipynb.")
else:
    if is_mcp != "true":
        print("\nWARNING: is_mcp_connection is not 'true'. Re-run the update cell.")
    if not url.endswith("/mcp"):
        print("WARNING: url does not end in /mcp. Set BASE_PATH to '/mcp' and re-run.")